# Section 1 · Your First AI Agent

In this notebook you build your first AI agent with the [Strands Agents SDK](https://strandsagents.com), powered by Amazon Bedrock.

You will:
1. Install the SDK
2. Pick a Bedrock model
3. Create an agent in one line
4. Give it your own tool with `@tool`
5. Give it a prebuilt tool
6. Shape its behaviour with a system prompt

**Before you start**
- AWS credentials configured (`aws configure`, AWS SSO, or a SageMaker execution role)
- Region set to one where Amazon Bedrock is available (this course uses `us-east-1`)

## Step 1: Install dependencies

In [1]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


## Step 2: Choose the model

Every example in this course reads the model from one variable. Change it here (or set the `BEDROCK_MODEL_ID` environment variable) to try a different model.

- `global.anthropic.claude-sonnet-4-6` — default, strong reasoning and tool use
- `global.anthropic.claude-haiku-4-5-20251001-v1:0` — faster and cheaper

In [2]:
import os

MODEL_ID = os.getenv("BEDROCK_MODEL_ID", "global.anthropic.claude-sonnet-4-6")
REGION = os.getenv("AWS_REGION", os.getenv("AWS_DEFAULT_REGION", "us-east-1"))
os.environ.setdefault("AWS_REGION", REGION)

print(f"Model : {MODEL_ID}")
print(f"Region: {REGION}")

Model : global.anthropic.claude-sonnet-4-6
Region: us-east-1


## Step 3: Create your first agent

In [3]:
from strands import Agent
from strands.models.bedrock import BedrockModel

model = BedrockModel(model_id=MODEL_ID, region_name=REGION)

# An agent = a model + (optional) tools + (optional) system prompt
agent = Agent(model=model)

response = agent("Explain Amazon Bedrock AgentCore in three sentences.")

NoCredentialsError: Unable to locate credentials

The agent streams its answer as it generates it. The returned object holds the full result — the final text, token usage, and the stop reason:

In [ ]:
print("Stop reason:", response.stop_reason)
print("Tokens used:", response.metrics.accumulated_usage)

Stop reason: end_turn
Tokens used: {'inputTokens': 20, 'outputTokens': 111, 'totalTokens': 131}


## Step 4: Give the agent a tool

A tool is just a Python function with:
- the `@tool` decorator
- type hints (so the model knows what arguments to send)
- a docstring (so the model knows *when* to use it)

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

from strands import tool


@tool
def current_time(timezone: str = "Asia/Kolkata") -> str:
    """Get the current date and time in a given IANA timezone.

    Args:
        timezone: IANA timezone name, for example "Asia/Kolkata" or "America/New_York".
    """
    return datetime.now(ZoneInfo(timezone)).strftime("%A, %d %B %Y %I:%M %p %Z")


agent = Agent(model=model, tools=[current_time])
agent("What time is it right now in Bangalore and in New York?")

I'll check the current time in both cities simultaneously!
Tool #1: current_time

Tool #2: current_time
Here's the current time in both cities:

- 🇮🇳 **Bangalore (IST):** Wednesday, 14 October 2026 at 6:47 AM
- 🇺🇸 **New York (EDT):** Tuesday, 13 October 2026 at 9:17 PM

Bangalore is **9 hours and 30 minutes ahead** of New York, which is why they're even on different calendar days right now!

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Here's the current time in both cities:\n\n- 🇮🇳 **Bangalore (IST):** Wednesday, 14 October 2026 at 6:47 AM\n- 🇺🇸 **New York (EDT):** Tuesday, 13 October 2026 at 9:17 PM\n\nBangalore is **9 hours and 30 minutes ahead** of New York, which is why they're even on different calendar days right now!"}], 'metadata': {'usage': {'inputTokens': 820, 'outputTokens': 113, 'totalTokens': 933}, 'metrics': {'latencyMs': 1641, 'timeToFirstByteMs': 1057}}, 'tracking_id': '870e3112-f5ce-4cbd-a992-0cff34fa3bc6'}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'current_time': ToolMetrics(tool={'toolUseId': 'tooluse_xLSCWBno7Jg1eFM2DGYOvz', 'name': 'current_time', 'input': {'timezone': 'America/New_York'}}, call_count=2, success_count=2, error_count=0, total_time=0.005872249603271484)}, cycle_durations=[1.588059902191162, 1.8885912895202637], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_l

## Step 5: Use a prebuilt tool

`strands-agents-tools` ships ready-made tools. Here the agent gets a calculator so it does not have to do arithmetic "in its head".

In [ ]:
from strands_tools import calculator

agent = Agent(model=model, tools=[calculator, current_time])
agent("What is 18% GST on a laptop that costs Rs 84,999? Give me the final price too.")


Tool #1: calculator

Tool #2: calculator


DEPRECATION WARNING: calculator is deprecated. This warning becomes an error log in v0.9.0. To achieve similar functionality, use the shell tool vended by strands-agents (from strands.vended_tools import shell). This does change the security boundary: calculator only ever evaluated an expression checked against an AST allowlist, while shell executes arbitrary commands, so review it against your threat model before switching.
DEPRECATION WARNING: calculator is deprecated. This warning becomes an error log in v0.9.0. To achieve similar functionality, use the shell tool vended by strands-agents (from strands.vended_tools import shell). This does change the security boundary: calculator only ever evaluated an expression checked against an AST allowlist, while shell executes arbitrary commands, so review it against your threat model before switching.


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ 84999 * 18 / 100    │                                                                            │
│  │ Result    │ 15299.82            │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ 84999 * 1.18        │                                                                            │
│  │ Result    │ 100298.82           │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Here's the full GST breakdown for your laptop:

| Description | Amount |
|---|---|
| 💻 Laptop Base Price | ₹84,999.00 |
| 🧾 GST (18%) | ₹15,299.82 |
| ✅ **Final Price** | **₹1,00,298.82** |

### Quick Breakdown:
- **Original Price:** ₹84,999
- **GST Amount (18%):** ₹15,299.82
- **Final Price (inclusive of GST):** ₹1,00,298.82

> 💡 **Note:** The 18% GST on laptops includes **9% CGST + 9% SGST** (or 18% IGST for inter-state purchases).

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Here's the full GST breakdown for your laptop:\n\n| Description | Amount |\n|---|---|\n| 💻 Laptop Base Price | ₹84,999.00 |\n| 🧾 GST (18%) | ₹15,299.82 |\n| ✅ **Final Price** | **₹1,00,298.82** |\n\n### Quick Breakdown:\n- **Original Price:** ₹84,999\n- **GST Amount (18%):** ₹15,299.82\n- **Final Price (inclusive of GST):** ₹1,00,298.82\n\n> 💡 **Note:** The 18% GST on laptops includes **9% CGST + 9% SGST** (or 18% IGST for inter-state purchases)."}], 'metadata': {'usage': {'inputTokens': 2216, 'outputTokens': 196, 'totalTokens': 2412}, 'metrics': {'latencyMs': 2474, 'timeToFirstByteMs': 1101}}, 'tracking_id': 'ad51bc71-1ad7-4263-ac24-9bf010105513'}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'calculator': ToolMetrics(tool={'toolUseId': 'tooluse_JGMm9KCQhiDc5K7kFKYQSH', 'name': 'calculator', 'input': {'expression': '84999 * 1.18', 'mode': 'evaluate'}}, call_count=2, success_count=2, error_

## Step 6: Shape behaviour with a system prompt

The system prompt sets the agent's role, tone, and rules for every turn.

In [ ]:
agent = Agent(
    model=model,
    system_prompt=(
        "You are a patient tutor for engineers who are new to AI agents. "
        "Explain ideas with one real-world analogy and keep answers under 120 words."
    ),
)
agent("What is the difference between an LLM and an AI agent?")

## LLM vs. AI Agent

**An LLM** is like a brilliant advisor sitting in a room — you ask a question, they give a great answer, but they stay in that room. They only *respond* to what you bring them.

**An AI agent** is like that same advisor who can also *pick up the phone, search the internet, book a meeting, and follow up tomorrow.* They take actions in the world across multiple steps to reach a goal.

**In short:**
- LLM = smart text generator (single response)
- AI Agent = LLM + tools + memory + the ability to plan and act autonomously over time

The LLM is typically the *brain inside* the agent.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': '## LLM vs. AI Agent\n\n**An LLM** is like a brilliant advisor sitting in a room — you ask a question, they give a great answer, but they stay in that room. They only *respond* to what you bring them.\n\n**An AI agent** is like that same advisor who can also *pick up the phone, search the internet, book a meeting, and follow up tomorrow.* They take actions in the world across multiple steps to reach a goal.\n\n**In short:**\n- LLM = smart text generator (single response)\n- AI Agent = LLM + tools + memory + the ability to plan and act autonomously over time\n\nThe LLM is typically the *brain inside* the agent.'}], 'metadata': {'usage': {'inputTokens': 56, 'outputTokens': 161, 'totalTokens': 217}, 'metrics': {'latencyMs': 2978, 'timeToFirstByteMs': 1096}}, 'tracking_id': 'd6fad2b7-0952-4960-bf70-80b4e50690dc'}, metrics=EventLoopMetrics(cycle_count=1, tool_metrics={}, cycle_durations=[3.220686197280884

: 

## What you just did

- Built an agent with one line of code
- Added your own tool and a prebuilt tool
- Controlled behaviour with a system prompt

Next, run the two scripts in this folder to see the same idea as plain Python files, including one built with LangGraph:

```bash
python hello_agent.py
python hello_langgraph_agent.py
```